# 04. ROS — VLM 지연과 판단 유지 시간 / 75분

**선행:** 01 규칙 이동, 03 VLM 손 추종

**이 노트북의 결과:** `LatestOnlyMailbox`로 최신 프레임만 보관하고 TTL 0.4·1·3초에서 이동 연속성과 정지 반응을 비교합니다. 다음 판단이 오지 않을 때에도 이전 판단을 유지하는 상한을 검증합니다.

[단원 README](README.md) · [용어집](GLOSSARY.md)


<!-- lesson-purpose:start -->
## 이 실습은 무엇을 위한 것인가요?

느린 판단을 얼마나 오래 사용할지 정하면 움직임의 연속성과 오래된 명령의 유지가 함께 달라집니다. 이번 실습은 처리속도를 무조건 높이는 대신 TTL만 바꾸어 두 효과를 관찰합니다. 최신 대기 프레임과 진행 중 추론은 다른 상태입니다.

### 전체 과정에서 내 코드가 하는 일

```text
최신 한 장 TODO
  ↓
같은 VLM·입력·제어 조건
  ↓
TTL 400/1000/3000ms
  ↓
이동·정지 시각
  ↓
지연과 연속성 비교
```

put/take·깊은 복사·대체 횟수 계약을 구현합니다. TTL 외 촬영 age·pose age·속도 조건은 유지하고 새 STOP 또는 목표 도달이 TTL보다 먼저 멈출 수 있음을 관찰합니다.

### 실행 전에 생각하고 결과로 설명하기

**질문:** TTL을 3000ms로 설정하면 반드시 3초 동안 더 이동하나요?

**설명:** 아닙니다. 유효한 판단을 허용하는 최대 시간일 뿐입니다. 새 STOP·목표 도달·오래된 pose 등 다른 조건으로 먼저 정지합니다. 움직인 시간과 입력 변경 뒤 정지 지연을 둘 다 기록해야 합니다.

이후의 TODO는 각 절에서 지정한 위치만 수정하고, 정답 해설의 이유를 자기 입력에 적용해 설명합니다. 예제 실행 성공·학생 함수 검증·실제 장치 관찰은 해당 절에서 요구하는 범위대로 나누어 기록합니다.
<!-- lesson-purpose:end -->


## 실습 시작 확인

현재 노트북의 Python 커널과 `/workspace` 자료 경로를 확인합니다. 새 커널이거나 커널을 재시작했다면 첫 설정·import 셀부터 순서대로 실행합니다. 이전 노트북의 변수·import는 자동으로 전달되지 않습니다. 저장한 노트북·소스와 `Data` 파일은 유지됩니다. 접속·재부팅 복구는 [Setup 안내](../01.Setup/README.md)를 참고합니다.

**4일차 환경:** `nepes-course:physical-ai-v1.0`, Python `/opt/course/bin/python`을 사용합니다. 카메라·GUI는 로컬 화면에서 함께 관찰하고 입력 프로세스는 하나만 실행합니다.

**VLM 실습 준비:** VLM 질의에는 공용 Ollama의 `http://127.0.0.1:11434` API와 `gemma4:e2b-it-qat` 모델이 필요합니다. 해당 실습의 연결 확인에서 실패하면 로컬 Setup 담당자에게 서버·모델 준비를 요청합니다. 순수 Python fixture와 YOLO 실습에는 Ollama가 필요하지 않습니다.

**작업 순서:** 아래 첫 설정 셀의 `ROOT`가 `/workspace`인지 확인 → 현재 노트북이 지정한 `exercises/*.py`를 Jupyter 편집기에서 저장 → `student = load_file(...)` 검증 셀을 다시 실행합니다. 새 커널에는 이전 노트북의 변수나 import가 없으므로 이 노트북 자체의 설정·fixture 셀을 사용합니다. 실행 중 보이는 출력은 학습 확인용이고 배포본에는 저장하지 않습니다. TODO 실패와 카메라·모델 환경 실패를 구분하세요.

## 슬라이드와 함께 진행하기

시작 안내는 [4일차 슬라이드 37쪽](https://docs.google.com/presentation/d/1M3huhBQOov8Du3WoRq0pTke0Aqfj33eLKloCLBtwOow/edit#slide=id.d4_s062) · [배포 PDF 37쪽](../슬라이드/4일차.pdf#page=37)입니다. 노트북의 설명·TODO·검증·실제 실행을 진행한 뒤, [4일차 슬라이드 38쪽](https://docs.google.com/presentation/d/1M3huhBQOov8Du3WoRq0pTke0Aqfj33eLKloCLBtwOow/edit#slide=id.d4_s035) · [배포 PDF 38쪽](../슬라이드/4일차.pdf#page=38)로 돌아가 38–49쪽에서 결과를 복습합니다. 쪽수는 표지를 포함한 실제 페이지 순서입니다.


## 정답 활용 안내

정답 확인 시점과 직접 시도 시간은 강사가 현장 진도·이해도에 맞춰 조절합니다. 검증 통과 전에도 강사 안내에 따라 정답을 참고할 수 있습니다. 정답과 해설을 비교한 뒤 자신의 TODO에 반영하고 다시 검증하세요. 정답 실행만으로 자신의 TODO가 완성되지는 않습니다.


## 이번 실습이 전체 흐름에서 맡는 역할

[01 규칙 이동](01_ros_gesture_action.ipynb) → [02 VLM 프롬프트 검증](02_vlm_decision.ipynb) → [03 VLM 손 추종](03_ros_vlm_action.ipynb) → [04 판단 유지시간](04_ros_vlm_latency.ipynb) → [05 YOLO detection·pose](05_yolo_detection_pose.ipynb) → [06 pose 좌표·LLM·ROS](06_ros_pose_llm.ipynb). [07 segmentation](07_yolo_segmentation.ipynb)은 선택 자습입니다.

**구현 실습과 실제 실행의 관계:** 지정한 `exercises/` 파일에 학생이 알고리즘을 구현하고 노트북 검증 셀로 평가합니다. `solutions/`는 별도 참고 정답이며, 참고 정답 검증만 실행해도 학생 TODO가 완성되는 것은 아닙니다. 카메라·ROS 데모는 제공된 실행용 구현을 사용합니다. 학생 파일 저장이 실행 중인 ROS 노드를 자동으로 바꾸지는 않습니다. 실시간 동작과 자신의 함수 검증을 각각 완료하세요.


## 시간 지도

지연·유효시간 10분 → mailbox TODO·검증 15분 → 고정 입력 경계 10분 → VLM TTL 실험 30분 → 기록 10분


## 필요한 앞 내용

[03 VLM 손 추종](03_ros_vlm_action.ipynb)에서 같은 프레임의 좌표와 판단을 연결했습니다. captured_ms는 촬영, issued_ms는 판단 완료 시각입니다. TTL은 판단 완료 후 승인 유지 상한입니다. 프레임 age 5000ms와 pose age 500ms 제한은 별도로 유지합니다.

아래 LatestOnlyMailbox TODO는 대기 프레임을 최신 한 장만 남깁니다. 대기열을 줄여도 진행 중인 VLM 요청이 즉시 끝나지는 않습니다.


In [ ]:
# 구분: 수업 핵심 셀
from pathlib import Path
import importlib.util, json, shutil, sys

def find_course_root():
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "09.Pseudo-VLA").is_dir():
            return candidate
    raise RuntimeError("09.Pseudo-VLA를 포함한 과정 루트를 찾지 못했습니다.")

ROOT = find_course_root()
LESSON = ROOT / "09.Pseudo-VLA"
if str(LESSON) not in sys.path:
    sys.path.insert(0, str(LESSON))
OPTIONAL = {name: bool(importlib.util.find_spec(name)) for name in ('cv2', 'ultralytics', 'rclpy', 'requests')}
print({"course_root": str(ROOT), "optional_live_dependencies": OPTIONAL})

def load_file(module_name, relative_path):
    path = LESSON / relative_path
    spec = importlib.util.spec_from_file_location(module_name, path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


## 다음 판단을 기다리는 동안 얼마나 더 이동할까?

<!-- procedure:lesson-33-c07d19:start -->
![다음 판단까지 이동 허용 시간](../교재그림/절차/lesson-33.svg)

예시: 다음 판단이 만료 후 도착하는 경우입니다. TTL·frame age 중 먼저 끝나는 한계에서 정지하며 새 STOP·도착·오류는 더 일찍 멈춥니다.
<!-- procedure:lesson-33-c07d19:end -->


VLM이 느리면 다음 판단 전에 승인이 만료되어 STOP 구간이 생깁니다. TTL만 400/1000/3000ms로 바꾸고 모델·입력·속도·프레임 age·pose 제한은 유지합니다.

Approval age, TTL, valid left를 함께 읽습니다. 남은 유효시간은 승인 TTL과 프레임 age 중 짧은 쪽입니다. 다른 검사나 목표 도착 때문에 더 일찍 멈출 수도 있습니다.

유지 시간은 이동 허용의 상한입니다. 새 판단은 즉시 교체하며 목표 도착·새 STOP·오류는 TTL이 남아 있어도 정지합니다. 이전 속도를 무조건 유지하거나 새 좌표에 옛 승인을 붙이지 않습니다.

가장 오래 움직이는 설정을 고르는 대신 연속성과 정지 지연을 함께 설명합니다.


In [ ]:
# 구분: 수업 핵심 셀
from common.contracts import Frame
frames = [Frame(f"f{i}", i*100, {"image":"fixture"}) for i in range(3)]
print(frames)


> **구분: 수업 핵심** — 표시된 시간 안에 강사와 함께 진행합니다.

## TODO — [exercises/latest_frame_contract.py](exercises/latest_frame_contract.py)

`put(frame)`은 대기 관측을 최신 한 장으로 교체합니다. `take()`는 그 한 장을 반환하고 비웁니다.
FIFO에 쌓이는 세 프레임과 최신 한 장이 어떻게 다른지 종이에 그린 뒤 구현합니다.

<!-- todo-review:requirements:start -->
### 작성 전에 확인할 입력·출력과 순서

**입력 → 출력:** `LatestOnlyMailbox.put(frame)`의 frame은 제공 `Frame(frame_id,captured_ms,payload)`입니다. 보관하면 True, 현재 대기 프레임보다 촬영 시각이 오래되면 False입니다. take()는 최신 Frame 하나를 반환하고 슬롯을 비우며, 빈 슬롯에서는 None입니다.
**작성 흐름 힌트:** 입력 Frame 타입 검사 → 대기 프레임과 captured_ms 비교 → 오래된 입력은 기존 슬롯 유지 → 같거나 새 시각이면 교체하고 replaced 증가 → deepcopy로 payload 공유 차단 → take에서도 보관 상태를 비움. 대기 프레임이 없으면 replaced를 올리지 않습니다.
**손으로 확인:** a(100),b(200),old(150)를 put → True,True,False; take는 b, 다음 take는 None, replaced=1. 프레임 ID 문자열의 사전순이 아니라 captured_ms로 비교합니다.
**설명할 지점:** “추론이 느릴 때 오래된 관측을 모두 처리하면 현재와 멀어집니다. 여기서는 처리량보다 다음에 볼 입력의 최신성을 선택합니다.”
<!-- todo-review:requirements:end -->

### TODO의 자료형·보관 상태·완료 기준

수정 파일은 `/workspace/09.Pseudo-VLA/exercises/latest_frame_contract.py`입니다. 시작 코드는 FIFO 목록으로 잘못 보관하므로 `__init__`, `put`, `take`를 함께 바꿉니다. Frame 자료형은 이 노트북 예제와 같이 `from common.contracts import Frame`으로 가져옵니다. Frame의 `frame_id`는 식별자, `captured_ms`는 촬영 밀리초, `payload`는 이미지나 fixture를 담은 객체입니다. `Frame` 이외 입력은 TypeError입니다.

저장 슬롯은 Frame 또는 None 하나입니다. `put`은 대기 중인 Frame의 촬영 시각과 비교해 더 오래된 것만 False로 거부하고 같거나 새 시각은 True로 받아 교체합니다. 빈 슬롯에서는 직전에 소비한 Frame보다 과거 시각이어도 받을 수 있습니다. 전역 최대 시각을 기억하는 과제가 아닙니다. 입력 payload는 deepcopy로 복사해 호출자가 나중에 바꿔도 영향을 받지 않도록 합니다.

`replaced`는 0으로 시작하고 **기존 대기 Frame을 새 Frame으로 교체할 때만** 1 증가합니다. `take()`는 Frame 하나를 반환하고 슬롯을 None으로 비우며 다시 호출하면 None입니다. 참고 정답은 추가로 `accepted`(받은 횟수), `discarded_older`(오래되어 거절한 횟수), `peek()`(슬롯을 비우지 않는 복사 조회)를 제공하므로 정답을 읽을 때 함께 구분하세요. 핵심 TODO 검증은 put/take/replaced의 계약을 확인합니다.

**성공 기준:** a(100)→b(200)→old(150)의 put은 True/True/False, take는 b 다음 None, replaced=1입니다. 같은 시각 c(200)는 b를 교체하며 payload 원본 수정은 보관값에 영향을 주지 않아야 합니다. 비교는 ID의 사전순이 아니라 captured_ms의 숫자순입니다.


In [ ]:
# 구분: 수업 핵심 셀
student = load_file("student_mailbox", "exercises/latest_frame_contract.py")
box = student.LatestOnlyMailbox()
for frame in frames: box.put(frame)
assert box.take().frame_id == "f2"
assert box.take() is None
assert box.replaced == 2
check = student.LatestOnlyMailbox()
a = Frame("a", 100, {"pixels": [1]})
b = Frame("b", 200, {"pixels": [2]})
assert check.put(a) is True
assert check.put(b) is True
assert check.put(Frame("old", 150, {})) is False
assert check.replaced == 1
b.payload["pixels"][0] = 99
assert check.take().payload["pixels"] == [2]
assert check.take() is None
assert check.put(Frame("first", 200, {})) is True
assert check.put(Frame("same-time", 200, {})) is True
assert check.take().frame_id == "same-time"
try:
    check.put({"frame_id": "not-a-Frame", "captured_ms": 300})
except TypeError:
    pass
else:
    raise AssertionError("Frame 타입을 검사하세요")
print("PASS latest only")


> **구분: 수업 핵심** — 표시된 시간 안에 강사와 함께 진행합니다.

## 참고 정답

강사가 안내하는 시점에 아래 접힌 정답 셀을 여세요. 셀은 `solutions/`의 독립적인 참고 구현이 같은 계약을 통과하는지 확인합니다. 바로 아래 정답 해설·접힌 코드에 입력 검사·경계 처리·반환값을 다시 제공합니다. 실제 카메라·ROS 공통 구현은 별도 runtime이며 참고 정답 실행과 학생 TODO 검증은 각각 확인합니다.


In [ ]:
# 구분: 수업 핵심 셀
reference = load_file("reference_mailbox", "solutions/latest_frame_contract.py")
box = reference.LatestOnlyMailbox()
for frame in frames: box.put(frame)
assert box.take().frame_id == "f2"
assert box.take() is None
print("PASS reference")


### 정답 해설 — 입력부터 결과까지

**정답 구현 위치:** [solutions/latest_frame_contract.py](solutions/latest_frame_contract.py)의 `LatestOnlyMailbox`입니다. 대기 frame은 최대 한 개입니다. f1 뒤 f2가 오면 대기 중이던 f1을 버리고 f2로 바꿉니다. take는 최신 frame을 반환하면서 슬롯을 비워 같은 frame을 다시 꺼내지 않게 합니다.

f1→f2→f3를 처리 전에 넣으면 take는 f3, 다음 take는 None입니다. 교체 횟수는 실제로 대기 frame을 교체한 횟수입니다. frame ID·시간의 유효성/순서 검사도 정답 본문과 경계 검증을 함께 확인하세요.

latest-only는 대기열 누적을 막지만 이미 진행 중인 느린 추론을 즉시 끝내지는 않습니다. 결과 수신 후에도 frame age·ID·TTL 검사가 필요합니다.

**확인 질문:** FIFO처럼 f1부터 꺼내면 입력 속도가 처리보다 빠를 때 판단이 얼마나 과거로 밀리나요?

**적용·재검증:** 정답을 읽고 지정된 `exercises/` 파일의 TODO에 반영해 저장합니다. 노트북의 학습자 검증 셀을 다시 실행하고 경계 입력도 확인합니다. 참고 정답의 검증 통과는 학생 파일의 통과를 대신하지 않습니다. 이후 기존 실행·저장 셀에서 자신의 결과 경로를 확인합니다. 정답 알고리즘과 보조 함수는 표시된 solutions 파일에서 직접 읽습니다. 실제 카메라·ROS 실행은 제공 runtime을 사용하므로 학생 TODO 변경이 자동 반영된다고 가정하지 않습니다.

<!-- todo-review:answer:start -->
<details>
<summary>실행 없이 읽는 참고 정답과 단계별 주석</summary>

[실제 구현 파일](solutions/latest_frame_contract.py)에서 가져온 코드입니다. 이 Markdown 블록은 실행되지 않습니다. 정답 알고리즘은 아래 파일 본문에 직접 구현되어 있습니다. 학생 파일에서는 기존 함수 이름과 인자를 유지하세요. 아래에 포함된 보조 함수·상수는 알고리즘을 읽기 위한 문맥입니다.

```python
"""참고 정답: 최신 프레임 하나만 보관하는 mailbox입니다.
오래된 입력 거부 → 새 입력으로 교체 → take에서 슬롯 비우기 순서입니다.

입력 검사부터 결과 반환까지 이 파일에서 직접 확인합니다.
실제 카메라·ROS 실행용 공통 구현과 분리한 학습용 정답입니다.
"""

from __future__ import annotations
from copy import deepcopy
# 공통 프레임 자료형만 공유하며 보관·교체·소비 로직은 직접 구현합니다.
from common.contracts import Frame

class LatestOnlyMailbox:
    """최대 한 프레임만 보유해 느린 추론 앞에 backlog가 생기지 않게 한다."""

    def __init__(self) -> None:
        self._frame: Frame | None = None
        self.accepted = 0
        self.replaced = 0
        self.discarded_older = 0

    def put(self, frame: Frame) -> bool:
        if not isinstance(frame, Frame):
            raise TypeError("frame은 Frame이어야 합니다")
        if self._frame is not None and frame.captured_ms < self._frame.captured_ms:
            self.discarded_older += 1
            return False
        if self._frame is not None:
            self.replaced += 1
        # 호출자가 payload를 나중에 수정해도 보관 중인 관측은 바뀌지 않습니다.
        self._frame = deepcopy(frame)
        self.accepted += 1
        return True

    def take(self) -> Frame | None:
        # 반환 전에 슬롯을 비워 동일 프레임이 두 번 소비되지 않게 합니다.
        frame = self._frame
        self._frame = None
        return deepcopy(frame)

    def peek(self) -> Frame | None:
        return deepcopy(self._frame)
```

</details>
<!-- todo-review:answer:end -->


## 실제 ROS 실행 — JupyterLab 터미널 A·B·C

<!-- procedure:ros-terminals-57bc75:start -->
![새 터미널 준비와 세 역할](../교재그림/절차/ros-terminals.svg)

1–3을 새 터미널마다 적용합니다. 4–6은 동시에 실행하는 역할이며 노드·토픽 이름은 해당 실습 명령을 따릅니다.
<!-- procedure:ros-terminals-57bc75:end -->


새 Terminal에서는 아래 블록의 첫 줄 bash부터 전체를 복사합니다. A·B·C 모두 같은 배정 Domain(예:21)을 사용합니다. 노트북 Python 셀에 붙이지 않습니다. 이미 환경을 설정한 같은 B에서 모드·TTL만 바꿀 때는 Ctrl+C 후 launch 명령만 실행합니다.

**처음 준비 또는 패키지를 찾지 못할 때 — 빌드:**
```bash
bash
bash /workspace/08.ROS/scripts/build_workspace.sh
bash /workspace/09.Pseudo-VLA/scripts/build_workspace.sh
```

**A — 카메라 입력:** 이미 A가 실행 중이면 그대로 유지합니다. 카메라 publisher는 하나만 실행합니다.
```bash
bash
source /opt/ros/humble/setup.bash
source /workspace/Data/build/08.ROS/install/setup.bash
source /workspace/Data/build/09.Pseudo-VLA/install/setup.bash
export ROS_DOMAIN_ID=21
ros2 run v4l2_camera v4l2_camera_node --ros-args -p video_device:=/dev/video0 -p image_size:="[640, 480]" -r image_raw:=/camera/image_raw
```

카메라 번호는 배정값입니다. 녹화 입력으로 비교할 때는 A를 Ctrl+C로 종료하고 같은 A에서 아래 명령을 실행합니다. 매 조건에서 같은 시작 지점으로 재생을 다시 시작합니다.
```bash
ros2 run nepes_ros_lab recorded_image_publisher --ros-args -p image_source:=/workspace/Data/videos/07.Mediapipe/person_open_palm.avi
```

**화면 읽기:** /vla/image_overlay의 초록색 점은 전진·회전 중, 빨간색 점은 좌표가 있지만 정지 중입니다. 좌표가 없으면 NO TARGET입니다. Target은 거북이 목표 좌표, age는 제어에 사용한 프레임 경과 시간입니다. 승인 경과 시간·TTL·남은 유효시간도 확인합니다. VLM 추론 중에는 이전 관측 화면이 유지됩니다. 최신 원본은 /camera/image_raw입니다. mirror:=true는 목표 x만 반전하고 영상과 점은 원본 위치를 유지합니다.

**B — 첫 TTL 조건:**
```bash
bash
source /opt/ros/humble/setup.bash
source /workspace/Data/build/08.ROS/install/setup.bash
source /workspace/Data/build/09.Pseudo-VLA/install/setup.bash
export ROS_DOMAIN_ID=21
ros2 launch nepes_vla_demo hand_follow.launch.py mode:=vlm \
  ttl_ms:=400 \
  vlm_model:=gemma4:e2b-it-qat \
  log_path:=/workspace/Data/logs/09.Pseudo-VLA/04-ttl-0400.jsonl
```

앞 조건을 Ctrl+C로 종료한 **같은 B에서** 실행합니다. turtlesim을 새로 시작해 초기 pose를 맞추고 녹화 입력도 같은 시작 지점으로 맞춥니다.
```bash
ros2 launch nepes_vla_demo hand_follow.launch.py mode:=vlm \
  ttl_ms:=1000 \
  vlm_model:=gemma4:e2b-it-qat \
  log_path:=/workspace/Data/logs/09.Pseudo-VLA/04-ttl-1000.jsonl
```

앞 조건을 Ctrl+C로 종료한 **같은 B에서** 실행합니다. turtlesim을 새로 시작해 초기 pose를 맞추고 녹화 입력도 같은 시작 지점으로 맞춥니다.
```bash
ros2 launch nepes_vla_demo hand_follow.launch.py mode:=vlm \
  ttl_ms:=3000 \
  vlm_model:=gemma4:e2b-it-qat \
  log_path:=/workspace/Data/logs/09.Pseudo-VLA/04-ttl-3000.jsonl
```

**C — 관측·판단·행동 확인:** 새 Terminal에서 전체를 실행합니다.
```bash
bash
source /opt/ros/humble/setup.bash
source /workspace/Data/build/08.ROS/install/setup.bash
source /workspace/Data/build/09.Pseudo-VLA/install/setup.bash
export ROS_DOMAIN_ID=21
ros2 topic echo /vla/action
```

연속 출력은 Ctrl+C로 끝낸 뒤, 같은 C에서 필요에 따라 ros2 topic echo /vla/perception 또는 ros2 topic echo /vla/decision을 실행합니다. frame_id·reason·TTL·승인 경과 시간을 영상과 비교합니다.

**종료:** A 중단 → C에서 STOP 확인 → B 종료 → C 종료. 다음 노트북으로 바로 이어갈 때는 A를 유지하고 B만 종료합니다. GUI 없는 자습은 B launch 앞에 QT_QPA_PLATFORM=offscreen, 인자에 with_image_view:=false를 사용합니다.

학생 TODO는 Python 검증 셀로 평가합니다. 실제 ROS는 제공 공통 구현을 실행하며 학생 파일 변경이 자동 반영되지는 않습니다.


## 완료 검사 — 같은 입력으로 3조건 비교

각 조건을 3회 반복하고 모델 적재 직후 첫 판단은 별도 기록합니다.

1. 중앙에서 떨어진 손 목표로 FOLLOW를 받아 이동 구간을 확보합니다.
2. 주먹 전환·손 제거·다른 위치 이동을 각각 따로 수행합니다. 원본 장면의 전환 시각을 관찰자가 기록합니다.
3. STOP 시각·이유를 기록합니다. 목표 도착과 TTL 만료를 구분합니다.
4. 이전 목표로 더 움직인 시간, 정지 구간, 전환 후 정지 반응을 비교합니다. 과거 관측 화면만으로 실제 손 전환 시각을 정하지 않습니다.
5. 다음 판단이 없을 때의 만료 경계는 아래 고정 입력 셀로 검증합니다.

| TTL(ms) | 반복 | 판단 지연 | 정지 구간 | 주먹·손 제거 후 정지 지연 | 이전 목표로 더 움직임 | STOP 이유 |
|---:|---:|---|---|---|---|---|
| 400 | 1–3 | 미측정 | 미측정 | 미측정 | 미측정 | 미측정 |
| 1000 | 1–3 | 미측정 | 미측정 | 미측정 | 미측정 | 미측정 |
| 3000 | 1–3 | 미측정 | 미측정 | 미측정 | 미측정 | 미측정 |

로그는 04-ttl-0400.jsonl, 04-ttl-1000.jsonl, 04-ttl-3000.jsonl입니다. 추가 반복은 log_path에 회차 이름을 붙여 구분합니다. 전환 시각을 기록하지 않았다면 반응 시간은 미측정입니다.


## 고정 입력으로 TTL 경계 확인

실제 VLM 측정이 아닙니다. 한 번의 FOLLOW 뒤 새 판단이 없고 pose만 갱신되는 조건입니다. 같은 issued_ms를 유지하며 각 TTL 경계에서 정지하는지 확인합니다.


In [ ]:
from common.hand_follow import follow_motion, rule_decision
obs = dict(frame_id="ttl-fixture", captured_ms=1000, center=[1.0, 0.5], state="NON_FIST")
dec = rule_decision(obs)
for ttl in (400, 1000, 3000):
    rows = []
    for elapsed in (0, ttl-1, ttl):
        action = follow_motion(obs, dec, (5.5, 5.5, 0), now_ms=1000+elapsed,
                               issued_ms=1000, pose_ms=1000+elapsed, ttl_ms=ttl)
        rows.append((elapsed, action["command"], action["reason"]))
    assert rows[-2][1] == "FOLLOW"
    assert rows[-1][1:] == ("STOP", "expired")
    print("TTL", ttl, rows)
stop_dec = {**dec, "command": "STOP", "hand_state": "FIST"}
assert follow_motion(obs, stop_dec, (5.5,5.5,0), now_ms=1100,
                     issued_ms=1100, pose_ms=1100, ttl_ms=3000)["command"] == "STOP"
print("PASS TTL boundaries and new STOP")


## 로그 요약과 결과 저장

파일마다 마지막 session만 요약합니다. 첫 판단 지연은 이후 판단과 분리합니다. 행동 기록 사이의 시간을 집계하므로 마지막 로그 이후는 포함하지 않습니다. 정답률·손 전환 후 반응은 자동 추정하지 않습니다. 아래 셀은 실제 결과가 없으면 미측정으로 저장합니다.


In [ ]:
from datetime import datetime, timezone
import uuid
from common.latency_report import summarize_log
reports = []
for ttl in (400, 1000, 3000):
    path = ROOT / "Data/logs/09.Pseudo-VLA" / f"04-ttl-{ttl:04d}.jsonl"
    report = summarize_log(path, expected_ttl_ms=ttl)
    reports.append(report)
    print(json.dumps(report, ensure_ascii=False, indent=2))
run = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:8]
out = ROOT / "Data/outputs/09.Pseudo-VLA/latency" / run
out.mkdir(parents=True, exist_ok=False)
(out / "summary.json").write_text(json.dumps(reports, ensure_ascii=False, indent=2)+"\n")
print("Saved:", out / "summary.json")




<!-- slide-sync:start -->
## 슬라이드로 돌아가기 — 결과 복습

[4일차 슬라이드 38쪽](https://docs.google.com/presentation/d/1M3huhBQOov8Du3WoRq0pTke0Aqfj33eLKloCLBtwOow/edit#slide=id.d4_s035) · [배포 PDF 38쪽](../슬라이드/4일차.pdf#page=38)에서 38–49쪽을 복습합니다. 자신의 TODO 검증과 실제 관측·미측정을 나누어 설명합니다.
<!-- slide-sync:end -->

## 다음 단계 — 이미지 대신 구조화된 좌표

다음 [05 YOLO detection·pose](05_yolo_detection_pose.ipynb)에서 사람 박스와 17개 관절을 추출하고, [06 pose·LLM·ROS](06_ros_pose_llm.ipynb)에서는 이미지 없이 전체 관절 좌표 JSON을 LLM에 보내 코를 추종합니다. TTL 실험에서 배운 프레임·유효시간 검사는 이 경로에도 유지됩니다.

> **구분: 수업 후 복습·자습** — 상세 원리와 확장 내용입니다.

## 자습
ROS simulated time을 사용하려면 카메라·pose·worker가 같은 clock을 써야 합니다. 이번 turtlesim 수업은 기본 wall clock을 사용합니다.

### FIFO와 최신 한 장 비교 — 설명용 시간표

아래는 프레임이 100ms마다 도착하고 한 번의 처리가 250ms 걸린다고 가정한 시간표입니다. 실제 카메라·VLM 측정값이 아닙니다. 도착과 처리가 같은 시각이면 도착을 먼저 반영한다고 가정합니다.

| 시각(ms) | 입력/처리 이벤트 | FIFO의 다음 대기 | 최신 한 장의 다음 대기 |
|---:|---|---|---|
| 0 | f0 도착, 두 방식 모두 f0 처리 시작 | 없음 | 없음 |
| 100 | f1 도착 | f1 | f1 |
| 200 | f2 도착 | f1, f2 | f2 (f1 교체) |
| 250 | f0 처리 종료, 다음 처리 시작 | f1 처리, f2 대기 | f2 처리, 대기 없음 |
| 300 | f3 도착 | f2, f3 | f3 |
| 400 | f4 도착 | f2, f3, f4 | f4 (f3 교체) |
| 500 | f5 도착, 두 번째 처리 종료 | f2 처리, f3~f5 대기 | f5 처리, 대기 없음 |

500ms에 새로 처리하는 프레임의 촬영 후 나이는 FIFO 300ms, 최신 한 장 0ms입니다. 최신 한 장도 이미 진행 중인 추론은 취소하지 않으므로 결과의 frame age·TTL을 별도로 검사합니다. 종이나 노트북 Markdown 셀에 다음 시점을 그려 설명하세요. 이 자습에는 별도의 FIFO/latest 실행 셀이나 `self-study/04-latest-worker.json` 제출 파일이 없습니다. 실제 관측 기록은 앞서 안내한 ROS 로그에서 확인합니다.
